# Transform STATS19 Raw Data to Clean Data

## Mục tiêu

Transform ba file dữ liệu STATS19 giai đoạn 2021–2025:

- Collisions
- Vehicles
- Casualties

Nguyên tắc:

- Chỉ xử lý những vấn đề đã phát hiện trong profiling.
- Không tự ý xóa bản ghi.
- Không tự ý suy diễn giá trị còn thiếu.
- Categorical codes được ánh xạ bằng Data Guide chính thức.
- Mỗi bước transform đều có validation trước khi xuất file.

In [86]:
# Cell 1 — Import và khai báo đường dẫn
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 150)

PROJECT_ROOT = Path(r"D:\OLAP")

RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
REFERENCE_DIR = PROJECT_ROOT / "Data" / "Reference"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"

COLLISIONS_RAW_PATH = RAW_DIR / "collisions_2021_2025.csv"
VEHICLES_RAW_PATH = RAW_DIR / "vehicles_2021_2025.csv"
CASUALTIES_RAW_PATH = RAW_DIR / "casualties_2021_2025.csv"

DATA_GUIDE_PATH = REFERENCE_DIR / "data_guide.csv"

COLLISIONS_CLEAN_PATH = PROCESSED_DIR / "collisions_clean.csv"
VEHICLES_CLEAN_PATH = PROCESSED_DIR / "vehicles_clean.csv"
CASUALTIES_CLEAN_PATH = PROCESSED_DIR / "casualties_clean.csv"

# Tạo thư mục Processed nếu chưa tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)

Raw directory: D:\OLAP\Data\Raw
Processed directory: D:\OLAP\Data\Processed


In [87]:
# Cell 2 — Kiểm tra các file đầu vào
input_files = {
    "Collisions Raw": COLLISIONS_RAW_PATH,
    "Vehicles Raw": VEHICLES_RAW_PATH,
    "Casualties Raw": CASUALTIES_RAW_PATH,
    "Data Guide": DATA_GUIDE_PATH
}

all_files_exist = True

for file_name, file_path in input_files.items():
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 ** 2)

        print(
            f"{file_name}: {file_path.name} "
            f"({size_mb:.2f} MB)"
        )
    else:
        all_files_exist = False
        print(f"Không tìm thấy {file_name}: {file_path}")

if all_files_exist:
    print("\nTất cả file đầu vào đều tồn tại.")
else:
    raise FileNotFoundError(
        "Có file đầu vào không tồn tại. "
        "Hãy kiểm tra lại đường dẫn trước khi tiếp tục."
    )

Collisions Raw: collisions_2021_2025.csv (93.14 MB)
Vehicles Raw: vehicles_2021_2025.csv (98.94 MB)
Casualties Raw: casualties_2021_2025.csv (49.95 MB)
Data Guide: data_guide.csv (0.12 MB)

Tất cả file đầu vào đều tồn tại.


In [88]:
# Cell 3 — Đọc và chuẩn hóa Data Guide
data_guide_raw = pd.read_csv(
    DATA_GUIDE_PATH,
    sep=";",
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False
)

data_guide_raw.columns = (
    data_guide_raw.columns
    .str.strip()
    .str.lower()
)

data_guide = data_guide_raw.rename(
    columns={
        "field name": "field_name",
        "code/format": "code"
    }
)

required_guide_columns = {
    "table",
    "field_name",
    "code",
    "label",
    "note"
}

missing_guide_columns = (
    required_guide_columns - set(data_guide.columns)
)

if missing_guide_columns:
    raise ValueError(
        f"Data Guide thiếu cột: {missing_guide_columns}"
    )

for column in required_guide_columns:
    data_guide[column] = data_guide[column].str.strip()

data_guide["table"] = data_guide["table"].str.lower()
data_guide["field_name"] = data_guide["field_name"].str.lower()

print("Đọc Data Guide thành công")
print("Số dòng:", len(data_guide))

display(data_guide.head())

Đọc Data Guide thành công
Số dòng: 1821


,table,field_name,code,label,note
0,collision,collision_index,,,unique value for each collision. The collision_index combines the collision_year and collision_ref_no to form a unique ID. It can be used to join to Vehicle and Casualty
1,collision,collision_year,,,
2,collision,collision_ref_no,,,"In year id used by the police to reference a collision. It is not unique outside of the year, use collision_index for linking to other years"
3,collision,location_easting_osgr,,,Null if not known
4,collision,location_northing_osgr,,,Null if not known


In [89]:
# Cell 4 — Hàm chuẩn hóa mã
def normalize_code(value):
    """
    Chuẩn hóa mã trước khi đối chiếu với Data Guide.

    Ví dụ:
    1, 1.0 và '1' đều được chuyển thành chuỗi '1'.
    """

    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if text == "":
        return ""

    try:
        number = float(text)

        if number.is_integer():
            return str(int(number))

    except ValueError:
        pass

    return text


data_guide["code"] = data_guide["code"].map(normalize_code)

lookup_mapping = data_guide[
    (data_guide["table"] != "") &
    (data_guide["field_name"] != "") &
    (data_guide["code"] != "") &
    (data_guide["label"] != "")
].copy()

mapping_conflicts = (
    lookup_mapping
    .groupby(["table", "field_name", "code"])["label"]
    .nunique()
    .reset_index(name="label_count")
    .query("label_count > 1")
)

if not mapping_conflicts.empty:
    display(mapping_conflicts)

    raise ValueError(
        "Data Guide có mã ánh xạ tới nhiều label khác nhau."
    )

lookup_mapping = lookup_mapping.drop_duplicates(
    subset=["table", "field_name", "code"],
    keep="first"
)

print("Lookup mapping hợp lệ")
print("Số mapping có thể sử dụng:", len(lookup_mapping))

Lookup mapping hợp lệ
Số mapping có thể sử dụng: 1774


In [90]:
supplemental_local_authority_mapping = pd.DataFrame([
    {
        "table": "collision",
        "field_name": "local_authority_ons_district",
        "code": "E07000246",
        "label": "Somerset West and Taunton",
        "note": (
            "Supplemented from official ONS geography. "
            "Historical local authority code."
        )
    },
    {
        "table": "collision",
        "field_name": "local_authority_ons_district",
        "code": "E08000038",
        "label": "Barnsley",
        "note": (
            "Supplemented from official ONS geography. "
            "New code following the 2025 boundary change."
        )
    },
    {
        "table": "collision",
        "field_name": "local_authority_ons_district",
        "code": "E08000039",
        "label": "Sheffield",
        "note": (
            "Supplemented from official ONS geography. "
            "New code following the 2025 boundary change."
        )
    }
], dtype="string")

lookup_mapping = pd.concat(
    [
        lookup_mapping,
        supplemental_local_authority_mapping
    ],
    ignore_index=True
)

# Nếu chạy lại cell nhiều lần, không để mapping bị nhân đôi
lookup_mapping = lookup_mapping.drop_duplicates(
    subset=["table", "field_name", "code"],
    keep="first"
)

print("Tổng số mapping hiện tại:", len(lookup_mapping))

Tổng số mapping hiện tại: 1777


# 1. Transform Collisions

Grain: mỗi dòng đại diện cho một vụ va chạm.

Số dòng kỳ vọng: 513,801.

Các bước:

1. Chỉ đọc 17 cột cần thiết.
2. Kiểm tra khóa `collision_index`.
3. Chuẩn hóa ngày và giờ.
4. Kiểm tra tọa độ.
5. Ánh xạ categorical codes bằng Data Guide.
6. Xử lý riêng `speed_limit`.
7. Validation và xuất file Clean.

In [91]:
# Cell 5 — Đọc 17 cột Collisions
COLLISION_COLUMNS = [
    "collision_index",
    "date",
    "time",
    "longitude",
    "latitude",
    "local_authority_ons_district",
    "collision_severity",
    "first_road_class",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area"
]

COLLISION_DTYPES = {
    "collision_index": "string",
    "date": "string",
    "time": "string",
    "longitude": "Float64",
    "latitude": "Float64",
    "local_authority_ons_district": "string",
    "collision_severity": "string",
    "first_road_class": "string",
    "road_type": "string",
    "speed_limit": "string",
    "junction_detail": "string",
    "junction_control": "string",
    "pedestrian_crossing": "string",
    "light_conditions": "string",
    "weather_conditions": "string",
    "road_surface_conditions": "string",
    "urban_or_rural_area": "string"
}

collisions_raw = pd.read_csv(
    COLLISIONS_RAW_PATH,
    usecols=COLLISION_COLUMNS,
    dtype=COLLISION_DTYPES,
    low_memory=False
)

print("Đã đọc Collisions Raw")
print("Số dòng:", len(collisions_raw))
print("Số cột:", len(collisions_raw.columns))

display(collisions_raw.head())

Đã đọc Collisions Raw
Số dòng: 513801
Số cột: 17


,collision_index,longitude,latitude,collision_severity,date,time,local_authority_ons_district,first_road_class,road_type,speed_limit,junction_detail,junction_control,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,urban_or_rural_area
0,202517M102225,-1.23119,54.56929,3,15/02/2025,19:15,E06000002,6,6,30,16,4,0,4,1,1,1
1,202417S111924,-1.31046,54.57776,2,22/10/2024,14:48,E06000004,3,6,30,0,-1,0,1,1,1,1
2,2025111687011,-1.57237,54.53082,3,19/12/2025,06:40,E06000005,6,6,30,13,4,0,4,2,2,1
3,2025070326701,-2.72862,53.34161,1,22/04/2025,20:03,E06000006,4,6,30,0,-1,0,1,1,1,1
4,2025070353559,-2.73589,53.34121,3,28/04/2025,17:40,E06000006,4,6,30,16,4,13,1,1,1,1


In [92]:
# Cell 6 — Tạo bản Clean và kiểm tra khóa
collisions_clean = collisions_raw.copy()

# Loại khoảng trắng thừa trong khóa
collisions_clean["collision_index"] = (
    collisions_clean["collision_index"]
    .str.strip()
)

missing_collision_keys = (
    collisions_clean["collision_index"].isna() |
    collisions_clean["collision_index"].eq("")
).sum()

duplicate_collision_keys = (
    collisions_clean["collision_index"]
    .duplicated()
    .sum()
)

print("Khóa bị thiếu:", missing_collision_keys)
print("Khóa bị trùng:", duplicate_collision_keys)

if missing_collision_keys > 0:
    raise ValueError(
        "collision_index có giá trị NULL hoặc rỗng."
    )

if duplicate_collision_keys > 0:
    raise ValueError(
        "collision_index không unique."
    )

print("collision_index hợp lệ và unique")

Khóa bị thiếu: 0
Khóa bị trùng: 0
collision_index hợp lệ và unique


In [93]:
# Cell 7 — Chuẩn hóa ngày và giờ
# ---------- DATE ----------

parsed_collision_date = pd.to_datetime(
    collisions_clean["date"].str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

invalid_date_count = parsed_collision_date.isna().sum()

if invalid_date_count > 0:
    invalid_dates = collisions_clean.loc[
        parsed_collision_date.isna(),
        ["collision_index", "date"]
    ]

    display(invalid_dates.head(20))

    raise ValueError(
        f"Có {invalid_date_count} giá trị date không hợp lệ."
    )

collision_years = sorted(
    parsed_collision_date.dt.year.unique().tolist()
)

expected_years = [2021, 2022, 2023, 2024, 2025]

if collision_years != expected_years:
    raise ValueError(
        f"Phạm vi năm không hợp lệ: {collision_years}"
    )

# CSV không lưu kiểu date nên chuẩn hóa thành ISO YYYY-MM-DD
collisions_clean["date"] = (
    parsed_collision_date.dt.strftime("%Y-%m-%d")
)


# ---------- TIME ----------

parsed_collision_time = pd.to_datetime(
    collisions_clean["time"].str.strip(),
    format="%H:%M",
    errors="coerce"
)

invalid_time_count = parsed_collision_time.isna().sum()

if invalid_time_count > 0:
    invalid_times = collisions_clean.loc[
        parsed_collision_time.isna(),
        ["collision_index", "time"]
    ]

    display(invalid_times.head(20))

    raise ValueError(
        f"Có {invalid_time_count} giá trị time không hợp lệ."
    )

collisions_clean["time"] = (
    parsed_collision_time.dt.strftime("%H:%M")
)

print("Date đã chuẩn hóa thành YYYY-MM-DD")
print("Time đã chuẩn hóa thành HH:MM")
print("Các năm:", collision_years)

Date đã chuẩn hóa thành YYYY-MM-DD
Time đã chuẩn hóa thành HH:MM
Các năm: [2021, 2022, 2023, 2024, 2025]


In [94]:
# Cell 8 — Kiểm tra tọa độ
longitude_missing = collisions_clean["longitude"].isna()
latitude_missing = collisions_clean["latitude"].isna()

coordinate_pair_mismatch = (
    longitude_missing != latitude_missing
).sum()

if coordinate_pair_mismatch > 0:
    mismatched_coordinates = collisions_clean.loc[
        longitude_missing != latitude_missing,
        [
            "collision_index",
            "longitude",
            "latitude"
        ]
    ]

    display(mismatched_coordinates.head(20))

    raise ValueError(
        "Có dòng chỉ thiếu longitude hoặc chỉ thiếu latitude."
    )

invalid_longitude = (
    collisions_clean["longitude"].notna() &
    ~collisions_clean["longitude"].between(-180, 180)
)

invalid_latitude = (
    collisions_clean["latitude"].notna() &
    ~collisions_clean["latitude"].between(-90, 90)
)

if invalid_longitude.any() or invalid_latitude.any():
    raise ValueError(
        "Có tọa độ nằm ngoài phạm vi địa lý hợp lệ."
    )

zero_coordinate_pairs = (
    collisions_clean["longitude"].eq(0) &
    collisions_clean["latitude"].eq(0)
).sum()

print("Số cặp tọa độ bị thiếu:", longitude_missing.sum())
print("Số cặp tọa độ (0, 0):", zero_coordinate_pairs)
print("Tọa độ hợp lệ; các giá trị thiếu được giữ là NULL")

Số cặp tọa độ bị thiếu: 53
Số cặp tọa độ (0, 0): 0
Tọa độ hợp lệ; các giá trị thiếu được giữ là NULL


In [95]:
# Cell 9 — Tạo hàm ánh xạ code sang label
def map_code_to_label(
    dataframe,
    table_name,
    column_name
):
    """
    Ánh xạ một categorical column từ code sang label
    bằng Data Guide.

    Nếu gặp code không có trong Data Guide,
    hàm sẽ dừng thay vì tự đoán.
    """

    normalized_codes = (
        dataframe[column_name]
        .astype("string")
        .map(normalize_code)
    )

    field_mapping_table = lookup_mapping[
        (lookup_mapping["table"] == table_name) &
        (lookup_mapping["field_name"] == column_name)
    ][
        ["code", "label"]
    ].drop_duplicates()

    if field_mapping_table.empty:
        raise ValueError(
            f"Không tìm thấy mapping cho "
            f"{table_name}.{column_name}"
        )

    field_mapping = dict(
        zip(
            field_mapping_table["code"],
            field_mapping_table["label"]
        )
    )

    raw_codes = set(
        normalized_codes.dropna().unique().tolist()
    )

    guide_codes = set(field_mapping.keys())

    unmapped_codes = sorted(raw_codes - guide_codes)

    if unmapped_codes:
        raise ValueError(
            f"{table_name}.{column_name} có mã "
            f"không tồn tại trong Data Guide: "
            f"{unmapped_codes}"
        )

    mapped_values = normalized_codes.map(field_mapping)

    if mapped_values.isna().sum() != normalized_codes.isna().sum():
        raise ValueError(
            f"Phát sinh NULL ngoài dự kiến khi mapping "
            f"{table_name}.{column_name}"
        )

    return mapped_values.astype("string")

In [96]:
# Cell 10 — Ánh xạ các cột categorical
# Giữ code địa lý và tạo thêm cột tên địa phương
collisions_clean["local_authority_ons_district"] = (
    collisions_clean["local_authority_ons_district"]
    .str.strip()
)

collisions_clean[
    "local_authority_ons_district_name"
] = map_code_to_label(
    dataframe=collisions_clean,
    table_name="collision",
    column_name="local_authority_ons_district"
)


# Các cột này được thay code bằng label
COLLISION_LABEL_COLUMNS = [
    "collision_severity",
    "first_road_class",
    "road_type",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area"
]

mapping_summary = [{
    "Column": "local_authority_ons_district",
    "Output": "local_authority_ons_district_name",
    "Raw_Distinct": (
        collisions_clean["local_authority_ons_district"]
        .nunique()
    ),
    "Clean_Distinct": (
        collisions_clean["local_authority_ons_district_name"]
        .nunique()
    ),
    "Status": "Code retained; name added"
}]

for column in COLLISION_LABEL_COLUMNS:
    raw_distinct_count = (
        collisions_clean[column]
        .nunique(dropna=True)
    )

    collisions_clean[column] = map_code_to_label(
        dataframe=collisions_clean,
        table_name="collision",
        column_name=column
    )

    clean_distinct_count = (
        collisions_clean[column]
        .nunique(dropna=True)
    )

    mapping_summary.append({
        "Column": column,
        "Output": column,
        "Raw_Distinct": raw_distinct_count,
        "Clean_Distinct": clean_distinct_count,
        "Status": "Code replaced by label"
    })

mapping_summary = pd.DataFrame(mapping_summary)

print("Đã ánh xạ categorical codes sang label")
display(mapping_summary)

Đã ánh xạ categorical codes sang label


,Column,Output,Raw_Distinct,Clean_Distinct,Status
0,local_authority_ons_district,local_authority_ons_district_name,370,368,Code retained; name added
1,collision_severity,collision_severity,3,3,Code replaced by label
2,first_road_class,first_road_class,7,7,Code replaced by label
3,road_type,road_type,6,6,Code replaced by label
4,junction_detail,junction_detail,8,8,Code replaced by label
5,junction_control,junction_control,6,6,Code replaced by label
6,pedestrian_crossing,pedestrian_crossing,10,10,Code replaced by label
7,light_conditions,light_conditions,6,6,Code replaced by label
8,weather_conditions,weather_conditions,10,10,Code replaced by label
9,road_surface_conditions,road_surface_conditions,7,7,Code replaced by label


In [97]:
# Cell 11 — Xử lý riêng speed_limit
speed_limit_raw = (
    collisions_clean["speed_limit"]
    .astype("string")
    .str.strip()
)

speed_limit_numeric = pd.to_numeric(
    speed_limit_raw,
    errors="coerce"
)

conversion_failed = (
    speed_limit_raw.notna() &
    speed_limit_numeric.isna()
)

if conversion_failed.any():
    invalid_speed_values = sorted(
        speed_limit_raw[conversion_failed]
        .unique()
        .tolist()
    )

    raise ValueError(
        f"speed_limit có giá trị không chuyển được sang số: "
        f"{invalid_speed_values}"
    )

# -1 không phải tốc độ thật
speed_limit_numeric = speed_limit_numeric.mask(
    speed_limit_numeric.eq(-1),
    pd.NA
)

allowed_speed_limits = {
    20, 30, 40, 50, 60, 70
}

actual_speed_limits = set(
    speed_limit_numeric
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

unexpected_speed_limits = (
    actual_speed_limits - allowed_speed_limits
)

if unexpected_speed_limits:
    raise ValueError(
        f"Xuất hiện speed_limit ngoài phạm vi dự kiến: "
        f"{sorted(unexpected_speed_limits)}"
    )

collisions_clean["speed_limit"] = (
    speed_limit_numeric.astype("Int64")
)

print(
    "Số speed_limit được chuyển thành NULL:",
    collisions_clean["speed_limit"].isna().sum()
)

print(
    "Các speed_limit hợp lệ:",
    sorted(actual_speed_limits)
)

Số speed_limit được chuyển thành NULL: 3
Các speed_limit hợp lệ: [20, 30, 40, 50, 60, 70]


In [98]:
# Cell 12 — Validation cuối bảng Collisions
validation_results = []

validation_results.append({
    "Check": "Row count unchanged",
    "Expected": len(collisions_raw),
    "Actual": len(collisions_clean),
    "Passed": len(collisions_raw) == len(collisions_clean)
})

validation_results.append({
    "Check": "Column count",
    "Expected": len(COLLISION_COLUMNS) + 1,
    "Actual": len(collisions_clean.columns),
    "Passed": (
        len(collisions_clean.columns) ==
        len(COLLISION_COLUMNS) + 1
    )
})

validation_results.append({
    "Check": "Missing collision_index",
    "Expected": 0,
    "Actual": collisions_clean["collision_index"].isna().sum(),
    "Passed": (
        collisions_clean["collision_index"]
        .isna()
        .sum() == 0
    )
})

validation_results.append({
    "Check": "Duplicate collision_index",
    "Expected": 0,
    "Actual": (
        collisions_clean["collision_index"]
        .duplicated()
        .sum()
    ),
    "Passed": (
        collisions_clean["collision_index"]
        .duplicated()
        .sum() == 0
    )
})

validation_results.append({
    "Check": "Missing coordinate pairs",
    "Expected": 53,
    "Actual": collisions_clean["longitude"].isna().sum(),
    "Passed": (
        collisions_clean["longitude"]
        .isna()
        .sum() == 53
    )
})

validation_results.append({
    "Check": "Missing speed_limit",
    "Expected": 3,
    "Actual": collisions_clean["speed_limit"].isna().sum(),
    "Passed": (
        collisions_clean["speed_limit"]
        .isna()
        .sum() == 3
    )
})

validation_results.append({
    "Check": "Missing local authority code",
    "Expected": 0,
    "Actual": (
        collisions_clean[
            "local_authority_ons_district"
        ].isna().sum()
    ),
    "Passed": (
        collisions_clean[
            "local_authority_ons_district"
        ].isna().sum() == 0
    )
})


collision_validation = pd.DataFrame(
    validation_results
)

display(collision_validation)

if not collision_validation["Passed"].all():
    raise ValueError(
        "Collisions Clean không vượt qua validation."
    )

print("Collisions Clean vượt qua toàn bộ validation")

display(collisions_clean.head())
display(collisions_clean.dtypes.to_frame("Data_Type"))


,Check,Expected,Actual,Passed
0,Row count unchanged,513801,513801,True
1,Column count,18,18,True
2,Missing collision_index,0,0,True
3,Duplicate collision_index,0,0,True
4,Missing coordinate pairs,53,53,True
5,Missing speed_limit,3,3,True
6,Missing local authority code,0,0,True


Collisions Clean vượt qua toàn bộ validation


,collision_index,longitude,latitude,collision_severity,date,time,local_authority_ons_district,first_road_class,road_type,speed_limit,junction_detail,junction_control,pedestrian_crossing,light_conditions,weather_conditions,road_surface_conditions,urban_or_rural_area,local_authority_ons_district_name
0,202517M102225,-1.23119,54.56929,Slight,2025-02-15,19:15,E06000002,Unclassified,Single carriageway,30,Crossroads,Give way or uncontrolled,No physical crossing facility within 50m,Darkness - lights lit,Fine no high winds,Dry,Urban,Middlesbrough
1,202417S111924,-1.31046,54.57776,Serious,2024-10-22,14:48,E06000004,A,Single carriageway,30,Not at junction or within 20 metres,Data missing or out of range,No physical crossing facility within 50m,Daylight,Fine no high winds,Dry,Urban,Stockton-on-Tees
2,2025111687011,-1.57237,54.53082,Slight,2025-12-19,06:40,E06000005,Unclassified,Single carriageway,30,T or staggered junction,Give way or uncontrolled,No physical crossing facility within 50m,Darkness - lights lit,Raining no high winds,Wet or damp,Urban,Darlington
3,2025070326701,-2.72862,53.34161,Fatal,2025-04-22,20:03,E06000006,B,Single carriageway,30,Not at junction or within 20 metres,Data missing or out of range,No physical crossing facility within 50m,Daylight,Fine no high winds,Dry,Urban,Halton
4,2025070353559,-2.73589,53.34121,Slight,2025-04-28,17:40,E06000006,B,Single carriageway,30,Crossroads,Give way or uncontrolled,Zebra crossing,Daylight,Fine no high winds,Dry,Urban,Halton


,Data_Type
collision_index,string[python]
longitude,Float64
latitude,Float64
collision_severity,string[python]
date,object
time,object
local_authority_ons_district,string[python]
first_road_class,string[python]
road_type,string[python]
speed_limit,Int64


In [99]:
# Cell 13 — Xuất collisions_clean.csv
collisions_clean.to_csv(
    COLLISIONS_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

output_size_mb = (
    COLLISIONS_CLEAN_PATH.stat().st_size /
    (1024 ** 2)
)

print("Đã xuất Collisions Clean")
print("Đường dẫn:", COLLISIONS_CLEAN_PATH)
print("Số dòng:", len(collisions_clean))
print("Số cột:", len(collisions_clean.columns))
print(f"Kích thước: {output_size_mb:.2f} MB")

Đã xuất Collisions Clean
Đường dẫn: D:\OLAP\Data\Processed\collisions_clean.csv
Số dòng: 513801
Số cột: 18
Kích thước: 120.63 MB


# 2. Transform Vehicles

Grain: mỗi dòng đại diện cho một phương tiện tham gia trong một vụ va chạm.

Natural key:

- collision_index
- vehicle_reference

Các bước:

1. Đọc 8 cột cần thiết.
2. Kiểm tra composite key.
3. Kiểm tra quan hệ với Collisions.
4. Ánh xạ categorical codes bằng Data Guide.
5. Validation và xuất Vehicles Clean.

In [100]:
# Cell 14 — Đọc 8 cột Vehicles
VEHICLE_COLUMNS = [
    "collision_index",
    "vehicle_reference",
    "vehicle_type",
    "vehicle_manoeuvre",
    "age_band_of_driver",
    "sex_of_driver",
    "skidding_and_overturning",
    "vehicle_leaving_carriageway"
]

VEHICLE_DTYPES = {
    "collision_index": "string",
    "vehicle_reference": "Int64",
    "vehicle_type": "string",
    "vehicle_manoeuvre": "string",
    "age_band_of_driver": "string",
    "sex_of_driver": "string",
    "skidding_and_overturning": "string",
    "vehicle_leaving_carriageway": "string"
}

vehicles_raw = pd.read_csv(
    VEHICLES_RAW_PATH,
    usecols=VEHICLE_COLUMNS,
    dtype=VEHICLE_DTYPES,
    low_memory=False
)

print("Đã đọc Vehicles Raw")
print("Số dòng:", len(vehicles_raw))
print("Số cột:", len(vehicles_raw.columns))

if len(vehicles_raw) != 937_265:
    raise ValueError(
        f"Số dòng Vehicles không đúng kỳ vọng: "
        f"{len(vehicles_raw):,}"
    )

display(vehicles_raw.head())

Đã đọc Vehicles Raw
Số dòng: 937265
Số cột: 8


,collision_index,vehicle_reference,vehicle_type,vehicle_manoeuvre,skidding_and_overturning,vehicle_leaving_carriageway,sex_of_driver,age_band_of_driver
0,2021461109918,1,9,19,0,0,1,8
1,202131B041421,1,9,19,0,0,2,9
2,2021010310120,1,9,5,0,0,1,6
3,2021440395823,1,9,19,0,0,1,6
4,2021461102971,1,9,1,0,0,2,6


In [101]:
# Cell 15 — Kiểm tra khóa và quan hệ với Collisions
vehicles_clean = vehicles_raw.copy()

vehicles_clean["collision_index"] = (
    vehicles_clean["collision_index"]
    .str.strip()
)

missing_vehicle_key = (
    vehicles_clean["collision_index"].isna() |
    vehicles_clean["collision_index"].eq("") |
    vehicles_clean["vehicle_reference"].isna()
)

missing_vehicle_key_count = missing_vehicle_key.sum()

duplicate_vehicle_key_count = (
    vehicles_clean[
        ["collision_index", "vehicle_reference"]
    ]
    .duplicated()
    .sum()
)

orphan_vehicle_mask = (
    ~vehicles_clean["collision_index"].isin(
        collisions_clean["collision_index"]
    )
)

orphan_vehicle_count = orphan_vehicle_mask.sum()

print("Khóa bị thiếu:", missing_vehicle_key_count)
print("Composite key bị trùng:", duplicate_vehicle_key_count)
print("Vehicles không có Collisions tương ứng:", orphan_vehicle_count)

if missing_vehicle_key_count > 0:
    display(
        vehicles_clean.loc[
            missing_vehicle_key,
            ["collision_index", "vehicle_reference"]
        ].head(20)
    )

    raise ValueError(
        "Vehicles có thành phần khóa bị thiếu."
    )

if duplicate_vehicle_key_count > 0:
    duplicated_vehicle_rows = vehicles_clean[
        vehicles_clean[
            ["collision_index", "vehicle_reference"]
        ].duplicated(keep=False)
    ].sort_values(
        ["collision_index", "vehicle_reference"]
    )

    display(duplicated_vehicle_rows.head(20))

    raise ValueError(
        "Composite key của Vehicles không unique."
    )

if orphan_vehicle_count > 0:
    display(
        vehicles_clean.loc[
            orphan_vehicle_mask,
            ["collision_index", "vehicle_reference"]
        ].head(20)
    )

    raise ValueError(
        "Vehicles có collision_index không tồn tại "
        "trong Collisions."
    )

print("Khóa Vehicles hợp lệ")
print("Không có orphan vehicle")

Khóa bị thiếu: 0
Composite key bị trùng: 0
Vehicles không có Collisions tương ứng: 0
Khóa Vehicles hợp lệ
Không có orphan vehicle


In [102]:
# Cell 16 — Ánh xạ categorical codes
VEHICLE_LABEL_COLUMNS = [
    "vehicle_type",
    "vehicle_manoeuvre",
    "age_band_of_driver",
    "sex_of_driver",
    "skidding_and_overturning",
    "vehicle_leaving_carriageway"
]

vehicle_mapping_summary = []

for column in VEHICLE_LABEL_COLUMNS:
    raw_distinct_count = (
        vehicles_clean[column]
        .nunique(dropna=True)
    )

    vehicles_clean[column] = map_code_to_label(
        dataframe=vehicles_clean,
        table_name="vehicle",
        column_name=column
    )

    clean_distinct_count = (
        vehicles_clean[column]
        .nunique(dropna=True)
    )

    vehicle_mapping_summary.append({
        "Column": column,
        "Raw_Distinct": raw_distinct_count,
        "Clean_Distinct": clean_distinct_count,
        "Status": "Code replaced by label"
    })

vehicle_mapping_summary = pd.DataFrame(
    vehicle_mapping_summary
)

print("Đã ánh xạ Vehicles codes sang labels")
display(vehicle_mapping_summary)

Đã ánh xạ Vehicles codes sang labels


,Column,Raw_Distinct,Clean_Distinct,Status
0,vehicle_type,22,22,Code replaced by label
1,vehicle_manoeuvre,19,19,Code replaced by label
2,age_band_of_driver,12,12,Code replaced by label
3,sex_of_driver,4,4,Code replaced by label
4,skidding_and_overturning,8,8,Code replaced by label
5,vehicle_leaving_carriageway,11,11,Code replaced by label


In [103]:
# Cell 17 — Kiểm tra các nhãn đặc biệt
vehicle_special_pattern = (
    r"unknown|not known|missing|out of range"
)

vehicle_special_values = []

for column in VEHICLE_LABEL_COLUMNS:
    special_counts = (
        vehicles_clean.loc[
            vehicles_clean[column]
            .fillna("")
            .str.contains(
                vehicle_special_pattern,
                case=False,
                regex=True
            ),
            column
        ]
        .value_counts()
        .reset_index()
    )

    special_counts.columns = ["Label", "Count"]
    special_counts.insert(0, "Column", column)

    vehicle_special_values.append(special_counts)

vehicle_special_summary = pd.concat(
    vehicle_special_values,
    ignore_index=True
)

display(vehicle_special_summary)

,Column,Label,Count
0,vehicle_type,Data missing or out of range,7562
1,vehicle_type,Goods vehicle - unknown weight,3920
2,vehicle_type,Motorcycle - unknown cc,2988
3,vehicle_type,Unknown vehicle type (self rep only),617
4,vehicle_manoeuvre,unknown (self reported),74901
5,vehicle_manoeuvre,Data missing or out of range,16854
6,age_band_of_driver,Data missing or out of range,142240
7,sex_of_driver,Not known,114525
8,sex_of_driver,Data missing or out of range,7845
9,skidding_and_overturning,unknown (self reported),76624


In [104]:
# Cell 18 — Validation Vehicles Clean
vehicle_validation_results = []

vehicle_validation_results.append({
    "Check": "Row count unchanged",
    "Expected": len(vehicles_raw),
    "Actual": len(vehicles_clean),
    "Passed": len(vehicles_raw) == len(vehicles_clean)
})

vehicle_validation_results.append({
    "Check": "Expected row count",
    "Expected": 937_265,
    "Actual": len(vehicles_clean),
    "Passed": len(vehicles_clean) == 937_265
})

vehicle_validation_results.append({
    "Check": "Column count",
    "Expected": len(VEHICLE_COLUMNS),
    "Actual": len(vehicles_clean.columns),
    "Passed": (
        len(vehicles_clean.columns) ==
        len(VEHICLE_COLUMNS)
    )
})

vehicle_validation_results.append({
    "Check": "Missing composite key",
    "Expected": 0,
    "Actual": missing_vehicle_key_count,
    "Passed": missing_vehicle_key_count == 0
})

vehicle_validation_results.append({
    "Check": "Duplicate composite key",
    "Expected": 0,
    "Actual": (
        vehicles_clean[
            ["collision_index", "vehicle_reference"]
        ]
        .duplicated()
        .sum()
    ),
    "Passed": (
        vehicles_clean[
            ["collision_index", "vehicle_reference"]
        ]
        .duplicated()
        .sum() == 0
    )
})

vehicle_validation_results.append({
    "Check": "Orphan collision_index",
    "Expected": 0,
    "Actual": (
        ~vehicles_clean["collision_index"].isin(
            collisions_clean["collision_index"]
        )
    ).sum(),
    "Passed": (
        ~vehicles_clean["collision_index"].isin(
            collisions_clean["collision_index"]
        )
    ).sum() == 0
})

vehicle_validation_results.append({
    "Check": "Missing categorical labels",
    "Expected": 0,
    "Actual": (
        vehicles_clean[VEHICLE_LABEL_COLUMNS]
        .isna()
        .sum()
        .sum()
    ),
    "Passed": (
        vehicles_clean[VEHICLE_LABEL_COLUMNS]
        .isna()
        .sum()
        .sum() == 0
    )
})

vehicle_validation = pd.DataFrame(
    vehicle_validation_results
)

display(vehicle_validation)

if not vehicle_validation["Passed"].all():
    raise ValueError(
        "Vehicles Clean không vượt qua validation."
    )

print("Vehicles Clean vượt qua toàn bộ validation")

display(vehicles_clean.head())
display(vehicles_clean.dtypes.to_frame("Data_Type"))

,Check,Expected,Actual,Passed
0,Row count unchanged,937265,937265,True
1,Expected row count,937265,937265,True
2,Column count,8,8,True
3,Missing composite key,0,0,True
4,Duplicate composite key,0,0,True
5,Orphan collision_index,0,0,True
6,Missing categorical labels,0,0,True


Vehicles Clean vượt qua toàn bộ validation


,collision_index,vehicle_reference,vehicle_type,vehicle_manoeuvre,skidding_and_overturning,vehicle_leaving_carriageway,sex_of_driver,age_band_of_driver
0,2021461109918,1,Car,Going ahead,None,Did not leave carriageway,Male,46 - 55
1,202131B041421,1,Car,Going ahead,None,Did not leave carriageway,Female,56 - 65
2,2021010310120,1,Car,Moving off,None,Did not leave carriageway,Male,26 - 35
3,2021440395823,1,Car,Going ahead,None,Did not leave carriageway,Male,26 - 35
4,2021461102971,1,Car,Reversing,None,Did not leave carriageway,Female,26 - 35


,Data_Type
collision_index,string[python]
vehicle_reference,Int64
vehicle_type,string[python]
vehicle_manoeuvre,string[python]
skidding_and_overturning,string[python]
vehicle_leaving_carriageway,string[python]
sex_of_driver,string[python]
age_band_of_driver,string[python]


In [105]:
# Cell 19 — Xuất vehicles_clean.csv
vehicles_clean.to_csv(
    VEHICLES_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

vehicle_output_size_mb = (
    VEHICLES_CLEAN_PATH.stat().st_size /
    (1024 ** 2)
)

print("Đã xuất Vehicles Clean")
print("Đường dẫn:", VEHICLES_CLEAN_PATH)
print("Số dòng:", len(vehicles_clean))
print("Số cột:", len(vehicles_clean.columns))
print(f"Kích thước: {vehicle_output_size_mb:.2f} MB")

Đã xuất Vehicles Clean
Đường dẫn: D:\OLAP\Data\Processed\vehicles_clean.csv
Số dòng: 937265
Số cột: 8
Kích thước: 82.77 MB


# 3. Transform Casualties

Grain: mỗi dòng đại diện cho một người thương vong trong một vụ va chạm.

Natural key:

- collision_index
- casualty_reference

Foreign keys:

- collision_index → Collisions
- (collision_index, vehicle_reference) → Vehicles

Các bước:

1. Đọc 8 cột cần thiết.
2. Kiểm tra natural key.
3. Kiểm tra quan hệ với Collisions và Vehicles.
4. Ánh xạ categorical codes bằng Data Guide.
5. Validation và xuất Casualties Clean.

In [106]:
# Cell 20 — Đọc 8 cột Casualties
CASUALTY_COLUMNS = [
    "collision_index",
    "vehicle_reference",
    "casualty_reference",
    "casualty_class",
    "sex_of_casualty",
    "age_band_of_casualty",
    "casualty_severity",
    "casualty_type"
]

CASUALTY_DTYPES = {
    "collision_index": "string",
    "vehicle_reference": "Int64",
    "casualty_reference": "Int64",
    "casualty_class": "string",
    "sex_of_casualty": "string",
    "age_band_of_casualty": "string",
    "casualty_severity": "string",
    "casualty_type": "string"
}

casualties_raw = pd.read_csv(
    CASUALTIES_RAW_PATH,
    usecols=CASUALTY_COLUMNS,
    dtype=CASUALTY_DTYPES,
    low_memory=False
)

print("Đã đọc Casualties Raw")
print("Số dòng:", len(casualties_raw))
print("Số cột:", len(casualties_raw.columns))

if len(casualties_raw) != 652_821:
    raise ValueError(
        f"Số dòng Casualties không đúng kỳ vọng: "
        f"{len(casualties_raw):,}"
    )

display(casualties_raw.head())

Đã đọc Casualties Raw
Số dòng: 652821
Số cột: 8


,collision_index,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_band_of_casualty,casualty_severity,casualty_type
0,2021010287661,1,1,3,1,8,3,0
1,2021010293865,1,1,3,1,6,3,0
2,2021010298600,1,1,3,1,2,3,0
3,2021010302619,1,1,3,1,7,3,0
4,2021010304568,1,1,3,1,3,3,0


In [107]:
# Cell 21 — Kiểm tra khóa Casualties
casualties_clean = casualties_raw.copy()

casualties_clean["collision_index"] = (
    casualties_clean["collision_index"]
    .str.strip()
)

missing_casualty_key = (
    casualties_clean["collision_index"].isna() |
    casualties_clean["collision_index"].eq("") |
    casualties_clean["casualty_reference"].isna()
)

missing_casualty_key_count = (
    missing_casualty_key.sum()
)

duplicate_casualty_key_count = (
    casualties_clean[
        ["collision_index", "casualty_reference"]
    ]
    .duplicated()
    .sum()
)

print("Khóa Casualties bị thiếu:", missing_casualty_key_count)
print("Natural key bị trùng:", duplicate_casualty_key_count)

if missing_casualty_key_count > 0:
    display(
        casualties_clean.loc[
            missing_casualty_key,
            ["collision_index", "casualty_reference"]
        ].head(20)
    )

    raise ValueError(
        "Casualties có thành phần natural key bị thiếu."
    )

if duplicate_casualty_key_count > 0:
    duplicated_casualties = casualties_clean[
        casualties_clean[
            ["collision_index", "casualty_reference"]
        ].duplicated(keep=False)
    ].sort_values(
        ["collision_index", "casualty_reference"]
    )

    display(duplicated_casualties.head(20))

    raise ValueError(
        "Natural key của Casualties không unique."
    )

print("Natural key của Casualties hợp lệ")

Khóa Casualties bị thiếu: 0
Natural key bị trùng: 0
Natural key của Casualties hợp lệ


In [108]:
# Cell 22 — Kiểm tra quan hệ với Collisions và Vehicles
# ---------- Quan hệ với Collisions ----------

orphan_casualty_collision_mask = (
    ~casualties_clean["collision_index"].isin(
        collisions_clean["collision_index"]
    )
)

orphan_casualty_collision_count = (
    orphan_casualty_collision_mask.sum()
)


# ---------- Quan hệ với Vehicles ----------

vehicle_key_index = pd.MultiIndex.from_frame(
    vehicles_clean[
        ["collision_index", "vehicle_reference"]
    ]
)

casualty_vehicle_key_index = pd.MultiIndex.from_frame(
    casualties_clean[
        ["collision_index", "vehicle_reference"]
    ]
)

orphan_casualty_vehicle_mask = (
    ~casualty_vehicle_key_index.isin(
        vehicle_key_index
    )
)

orphan_casualty_vehicle_count = (
    orphan_casualty_vehicle_mask.sum()
)

print(
    "Casualties không có Collision tương ứng:",
    orphan_casualty_collision_count
)

print(
    "Casualties không có Vehicle tương ứng:",
    orphan_casualty_vehicle_count
)

if orphan_casualty_collision_count > 0:
    display(
        casualties_clean.loc[
            orphan_casualty_collision_mask,
            [
                "collision_index",
                "casualty_reference"
            ]
        ].head(20)
    )

    raise ValueError(
        "Casualties có collision_index không tồn tại "
        "trong Collisions."
    )

if orphan_casualty_vehicle_count > 0:
    display(
        casualties_clean.loc[
            orphan_casualty_vehicle_mask,
            [
                "collision_index",
                "vehicle_reference",
                "casualty_reference"
            ]
        ].head(20)
    )

    raise ValueError(
        "Casualties tham chiếu đến Vehicle không tồn tại."
    )

print("Tất cả Casualties đều liên kết được với Collisions")
print("Tất cả Casualties đều liên kết được với Vehicles")

Casualties không có Collision tương ứng: 0
Casualties không có Vehicle tương ứng: 0
Tất cả Casualties đều liên kết được với Collisions
Tất cả Casualties đều liên kết được với Vehicles


In [109]:
# Cell 23 — Thống kê Vehicles có và không có Casualties
all_vehicle_keys = (
    vehicles_clean[
        ["collision_index", "vehicle_reference"]
    ]
    .drop_duplicates()
)

referenced_vehicle_keys = (
    casualties_clean[
        ["collision_index", "vehicle_reference"]
    ]
    .drop_duplicates()
)

vehicle_casualty_relationship = (
    all_vehicle_keys.merge(
        referenced_vehicle_keys.assign(
            Has_Casualty=True
        ),
        on=["collision_index", "vehicle_reference"],
        how="left"
    )
)

vehicle_casualty_relationship["Has_Casualty"] = (
    vehicle_casualty_relationship["Has_Casualty"]
    .fillna(False)
    .astype(bool)
)

vehicles_with_casualties = (
    vehicle_casualty_relationship["Has_Casualty"]
    .sum()
)

vehicles_without_casualties = (
    (~vehicle_casualty_relationship["Has_Casualty"])
    .sum()
)

vehicle_relationship_summary = pd.DataFrame([
    {
        "Group": "Vehicles with at least one casualty",
        "Count": vehicles_with_casualties,
        "Percent": round(
            vehicles_with_casualties /
            len(all_vehicle_keys) * 100,
            4
        )
    },
    {
        "Group": "Vehicles without casualty",
        "Count": vehicles_without_casualties,
        "Percent": round(
            vehicles_without_casualties /
            len(all_vehicle_keys) * 100,
            4
        )
    }
])

display(vehicle_relationship_summary)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_25036\637986822.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,Group,Count,Percent
0,Vehicles with at least one casualty,566945,60.4893
1,Vehicles without casualty,370320,39.5107


In [110]:
# Cell 24 — Ánh xạ categorical codes
CASUALTY_LABEL_COLUMNS = [
    "casualty_class",
    "sex_of_casualty",
    "age_band_of_casualty",
    "casualty_severity",
    "casualty_type"
]

casualty_mapping_summary = []

for column in CASUALTY_LABEL_COLUMNS:
    raw_distinct_count = (
        casualties_clean[column]
        .nunique(dropna=True)
    )

    casualties_clean[column] = map_code_to_label(
        dataframe=casualties_clean,
        table_name="casualty",
        column_name=column
    )

    clean_distinct_count = (
        casualties_clean[column]
        .nunique(dropna=True)
    )

    casualty_mapping_summary.append({
        "Column": column,
        "Raw_Distinct": raw_distinct_count,
        "Clean_Distinct": clean_distinct_count,
        "Status": "Code replaced by label"
    })

casualty_mapping_summary = pd.DataFrame(
    casualty_mapping_summary
)

print("Đã ánh xạ Casualties codes sang labels")
display(casualty_mapping_summary)

Đã ánh xạ Casualties codes sang labels


,Column,Raw_Distinct,Clean_Distinct,Status
0,casualty_class,3,3,Code replaced by label
1,sex_of_casualty,4,4,Code replaced by label
2,age_band_of_casualty,12,12,Code replaced by label
3,casualty_severity,3,3,Code replaced by label
4,casualty_type,23,23,Code replaced by label


In [111]:
# Cell 25 — Kiểm tra các nhãn đặc biệt
casualty_special_pattern = (
    r"unknown|not known|missing|out of range"
)

casualty_special_values = []

for column in CASUALTY_LABEL_COLUMNS:
    special_counts = (
        casualties_clean.loc[
            casualties_clean[column]
            .fillna("")
            .str.contains(
                casualty_special_pattern,
                case=False,
                regex=True
            ),
            column
        ]
        .value_counts()
        .reset_index()
    )

    special_counts.columns = ["Label", "Count"]
    special_counts.insert(0, "Column", column)

    casualty_special_values.append(special_counts)

casualty_special_summary = pd.concat(
    casualty_special_values,
    ignore_index=True
)

display(casualty_special_summary)

,Column,Label,Count
0,sex_of_casualty,Data missing or out of range,6304
1,sex_of_casualty,unknown (self reported),129
2,age_band_of_casualty,Data missing or out of range,14106
3,casualty_type,Data missing or out of range,5780
4,casualty_type,Motorcycle - unknown cc rider or passenger,1742
5,casualty_type,Goods vehicle (unknown weight) occupant,535
6,casualty_type,Unknown vehicle type (self rep only),33


In [112]:
# Cell 26 — Validation Casualties Clean
casualty_validation_results = []

casualty_validation_results.append({
    "Check": "Row count unchanged",
    "Expected": len(casualties_raw),
    "Actual": len(casualties_clean),
    "Passed": (
        len(casualties_raw) ==
        len(casualties_clean)
    )
})

casualty_validation_results.append({
    "Check": "Expected row count",
    "Expected": 652_821,
    "Actual": len(casualties_clean),
    "Passed": len(casualties_clean) == 652_821
})

casualty_validation_results.append({
    "Check": "Column count",
    "Expected": len(CASUALTY_COLUMNS),
    "Actual": len(casualties_clean.columns),
    "Passed": (
        len(casualties_clean.columns) ==
        len(CASUALTY_COLUMNS)
    )
})

casualty_validation_results.append({
    "Check": "Missing natural key",
    "Expected": 0,
    "Actual": missing_casualty_key_count,
    "Passed": missing_casualty_key_count == 0
})

casualty_validation_results.append({
    "Check": "Duplicate natural key",
    "Expected": 0,
    "Actual": (
        casualties_clean[
            ["collision_index", "casualty_reference"]
        ]
        .duplicated()
        .sum()
    ),
    "Passed": (
        casualties_clean[
            ["collision_index", "casualty_reference"]
        ]
        .duplicated()
        .sum() == 0
    )
})

casualty_validation_results.append({
    "Check": "Orphan Collision",
    "Expected": 0,
    "Actual": orphan_casualty_collision_count,
    "Passed": (
        orphan_casualty_collision_count == 0
    )
})

casualty_validation_results.append({
    "Check": "Orphan Vehicle",
    "Expected": 0,
    "Actual": orphan_casualty_vehicle_count,
    "Passed": (
        orphan_casualty_vehicle_count == 0
    )
})

casualty_validation_results.append({
    "Check": "Missing categorical labels",
    "Expected": 0,
    "Actual": (
        casualties_clean[CASUALTY_LABEL_COLUMNS]
        .isna()
        .sum()
        .sum()
    ),
    "Passed": (
        casualties_clean[CASUALTY_LABEL_COLUMNS]
        .isna()
        .sum()
        .sum() == 0
    )
})

casualty_validation_results.append({
    "Check": "Vehicles referenced by casualties",
    "Expected": 566_945,
    "Actual": len(referenced_vehicle_keys),
    "Passed": (
        len(referenced_vehicle_keys) ==
        566_945
    )
})

casualty_validation = pd.DataFrame(
    casualty_validation_results
)

display(casualty_validation)

if not casualty_validation["Passed"].all():
    raise ValueError(
        "Casualties Clean không vượt qua validation."
    )

print("Casualties Clean vượt qua toàn bộ validation")

display(casualties_clean.head())
display(casualties_clean.dtypes.to_frame("Data_Type"))

,Check,Expected,Actual,Passed
0,Row count unchanged,652821,652821,True
1,Expected row count,652821,652821,True
2,Column count,8,8,True
3,Missing natural key,0,0,True
4,Duplicate natural key,0,0,True
5,Orphan Collision,0,0,True
6,Orphan Vehicle,0,0,True
7,Missing categorical labels,0,0,True
8,Vehicles referenced by casualties,566945,566945,True


Casualties Clean vượt qua toàn bộ validation


,collision_index,vehicle_reference,casualty_reference,casualty_class,sex_of_casualty,age_band_of_casualty,casualty_severity,casualty_type
0,2021010287661,1,1,Pedestrian,Male,46 - 55,Slight,Pedestrian
1,2021010293865,1,1,Pedestrian,Male,26 - 35,Slight,Pedestrian
2,2021010298600,1,1,Pedestrian,Male,6 - 10,Slight,Pedestrian
3,2021010302619,1,1,Pedestrian,Male,36 - 45,Slight,Pedestrian
4,2021010304568,1,1,Pedestrian,Male,11 - 15,Slight,Pedestrian


,Data_Type
collision_index,string[python]
vehicle_reference,Int64
casualty_reference,Int64
casualty_class,string[python]
sex_of_casualty,string[python]
age_band_of_casualty,string[python]
casualty_severity,string[python]
casualty_type,string[python]


In [113]:
# Cell 27 — Xuất casualties_clean.csv
casualties_clean.to_csv(
    CASUALTIES_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

casualty_output_size_mb = (
    CASUALTIES_CLEAN_PATH.stat().st_size /
    (1024 ** 2)
)

print("Đã xuất Casualties Clean")
print("Đường dẫn:", CASUALTIES_CLEAN_PATH)
print("Số dòng:", len(casualties_clean))
print("Số cột:", len(casualties_clean.columns))
print(f"Kích thước: {casualty_output_size_mb:.2f} MB")

Đã xuất Casualties Clean
Đường dẫn: D:\OLAP\Data\Processed\casualties_clean.csv
Số dòng: 652821
Số cột: 8
Kích thước: 45.64 MB


# 4. Final Quality Assurance

Mục tiêu:

- Đọc lại ba file Clean từ ổ đĩa.
- Kiểm tra số dòng và số cột.
- Kiểm tra lại natural key.
- Kiểm tra quan hệ giữa ba file.
- Kiểm tra ngày, giờ, tọa độ và speed limit.
- Xác nhận dữ liệu sẵn sàng cho dimensional modeling.

In [ ]:
# Cell 28 — Giải phóng bộ nhớ và đọc lại file Clean
import gc

# Các file đã được xuất nên có thể xóa DataFrame cũ khỏi RAM
dataframes_to_clear = [
    "collisions_raw",
    "vehicles_raw",
    "casualties_raw",
    "collisions_clean",
    "vehicles_clean",
    "casualties_clean"
]

for dataframe_name in dataframes_to_clear:
    if dataframe_name in globals():
        del globals()[dataframe_name]

gc.collect()

print("Đã giải phóng các DataFrame cũ khỏi RAM")

✅ Đã giải phóng các DataFrame cũ khỏi RAM


In [115]:
COLLISIONS_CLEAN_DTYPES = {
    "collision_index": "string",
    "date": "string",
    "time": "string",
    "longitude": "Float64",
    "latitude": "Float64",
    "local_authority_ons_district": "string",
    "local_authority_ons_district_name": "string",
    "collision_severity": "string",
    "first_road_class": "string",
    "road_type": "string",
    "speed_limit": "Int64",
    "junction_detail": "string",
    "junction_control": "string",
    "pedestrian_crossing": "string",
    "light_conditions": "string",
    "weather_conditions": "string",
    "road_surface_conditions": "string",
    "urban_or_rural_area": "string"
}

VEHICLES_CLEAN_DTYPES = {
    "collision_index": "string",
    "vehicle_reference": "Int64",
    "vehicle_type": "string",
    "vehicle_manoeuvre": "string",
    "age_band_of_driver": "string",
    "sex_of_driver": "string",
    "skidding_and_overturning": "string",
    "vehicle_leaving_carriageway": "string"
}

CASUALTIES_CLEAN_DTYPES = {
    "collision_index": "string",
    "vehicle_reference": "Int64",
    "casualty_reference": "Int64",
    "casualty_class": "string",
    "sex_of_casualty": "string",
    "age_band_of_casualty": "string",
    "casualty_severity": "string",
    "casualty_type": "string"
}

collisions_disk = pd.read_csv(
    COLLISIONS_CLEAN_PATH,
    dtype=COLLISIONS_CLEAN_DTYPES,
    encoding="utf-8-sig",
    keep_default_na=False,
    na_values=[""]
)

vehicles_disk = pd.read_csv(
    VEHICLES_CLEAN_PATH,
    dtype=VEHICLES_CLEAN_DTYPES,
    encoding="utf-8-sig",
    keep_default_na=False,
    na_values=[""]
)

casualties_disk = pd.read_csv(
    CASUALTIES_CLEAN_PATH,
    dtype=CASUALTIES_CLEAN_DTYPES,
    encoding="utf-8-sig",
    keep_default_na=False,
    na_values=[""]
)

print("Đã đọc lại ba file Clean từ ổ đĩa")

✅ Đã đọc lại ba file Clean từ ổ đĩa


In [ ]:
# Cell 29 — Kiểm tra cấu trúc và khóa
structure_checks = [
    {
        "Table": "Collisions",
        "Check": "Row count",
        "Expected": 513_801,
        "Actual": len(collisions_disk)
    },
    {
        "Table": "Collisions",
        "Check": "Column count",
        "Expected": 18,
        "Actual": len(collisions_disk.columns)
    },
    {
        "Table": "Collisions",
        "Check": "Duplicate natural key",
        "Expected": 0,
        "Actual": (
            collisions_disk["collision_index"]
            .duplicated()
            .sum()
        )
    },
    {
        "Table": "Vehicles",
        "Check": "Row count",
        "Expected": 937_265,
        "Actual": len(vehicles_disk)
    },
    {
        "Table": "Vehicles",
        "Check": "Column count",
        "Expected": 8,
        "Actual": len(vehicles_disk.columns)
    },
    {
        "Table": "Vehicles",
        "Check": "Duplicate natural key",
        "Expected": 0,
        "Actual": (
            vehicles_disk[
                ["collision_index", "vehicle_reference"]
            ]
            .duplicated()
            .sum()
        )
    },
    {
        "Table": "Casualties",
        "Check": "Row count",
        "Expected": 652_821,
        "Actual": len(casualties_disk)
    },
    {
        "Table": "Casualties",
        "Check": "Column count",
        "Expected": 8,
        "Actual": len(casualties_disk.columns)
    },
    {
        "Table": "Casualties",
        "Check": "Duplicate natural key",
        "Expected": 0,
        "Actual": (
            casualties_disk[
                ["collision_index", "casualty_reference"]
            ]
            .duplicated()
            .sum()
        )
    }
]

structure_qa = pd.DataFrame(structure_checks)

structure_qa["Passed"] = (
    structure_qa["Expected"] ==
    structure_qa["Actual"]
)

display(structure_qa)

if not structure_qa["Passed"].all():
    raise ValueError(
        "Có file Clean không đạt kiểm tra cấu trúc."
    )

print("Số dòng, số cột và natural keys đều hợp lệ")

,Table,Check,Expected,Actual,Passed
0,Collisions,Row count,513801,513801,True
1,Collisions,Column count,18,18,True
2,Collisions,Duplicate natural key,0,0,True
3,Vehicles,Row count,937265,937265,True
4,Vehicles,Column count,8,8,True
5,Vehicles,Duplicate natural key,0,0,True
6,Casualties,Row count,652821,652821,True
7,Casualties,Column count,8,8,True
8,Casualties,Duplicate natural key,0,0,True


✅ Số dòng, số cột và natural keys đều hợp lệ


In [ ]:
# Cell 30 — Kiểm tra lại quan hệ giữa ba file
orphan_vehicles_disk = (
    ~vehicles_disk["collision_index"].isin(
        collisions_disk["collision_index"]
    )
).sum()

orphan_casualty_collisions_disk = (
    ~casualties_disk["collision_index"].isin(
        collisions_disk["collision_index"]
    )
).sum()

vehicle_keys_disk = pd.MultiIndex.from_frame(
    vehicles_disk[
        ["collision_index", "vehicle_reference"]
    ]
)

casualty_vehicle_keys_disk = pd.MultiIndex.from_frame(
    casualties_disk[
        ["collision_index", "vehicle_reference"]
    ]
)

orphan_casualty_vehicles_disk = (
    ~casualty_vehicle_keys_disk.isin(
        vehicle_keys_disk
    )
).sum()

relationship_qa = pd.DataFrame([
    {
        "Relationship": "Vehicles → Collisions",
        "Expected_Orphans": 0,
        "Actual_Orphans": orphan_vehicles_disk
    },
    {
        "Relationship": "Casualties → Collisions",
        "Expected_Orphans": 0,
        "Actual_Orphans": orphan_casualty_collisions_disk
    },
    {
        "Relationship": "Casualties → Vehicles",
        "Expected_Orphans": 0,
        "Actual_Orphans": orphan_casualty_vehicles_disk
    }
])

relationship_qa["Passed"] = (
    relationship_qa["Expected_Orphans"] ==
    relationship_qa["Actual_Orphans"]
)

display(relationship_qa)

if not relationship_qa["Passed"].all():
    raise ValueError(
        "Có quan hệ giữa các file Clean không hợp lệ."
    )

print("Các quan hệ giữa ba file Clean đều hợp lệ")

,Relationship,Expected_Orphans,Actual_Orphans,Passed
0,Vehicles → Collisions,0,0,True
1,Casualties → Collisions,0,0,True
2,Casualties → Vehicles,0,0,True


✅ Các quan hệ giữa ba file Clean đều hợp lệ


In [118]:
# Cell 31 — Kiểm tra nội dung Collisions
qa_dates = pd.to_datetime(
    collisions_disk["date"],
    format="%Y-%m-%d",
    errors="coerce"
)

qa_times = pd.to_datetime(
    collisions_disk["time"],
    format="%H:%M",
    errors="coerce"
)

coordinate_pair_mismatch_disk = (
    collisions_disk["longitude"].isna() !=
    collisions_disk["latitude"].isna()
).sum()

actual_speed_limits_disk = set(
    collisions_disk["speed_limit"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

content_qa = pd.DataFrame([
    {
        "Check": "Invalid ISO date",
        "Expected": 0,
        "Actual": qa_dates.isna().sum()
    },
    {
        "Check": "Invalid HH:MM time",
        "Expected": 0,
        "Actual": qa_times.isna().sum()
    },
    {
        "Check": "Years outside 2021–2025",
        "Expected": 0,
        "Actual": (
            ~qa_dates.dt.year.isin(
                [2021, 2022, 2023, 2024, 2025]
            )
        ).sum()
    },
    {
        "Check": "Missing coordinate pairs",
        "Expected": 53,
        "Actual": (
            collisions_disk["longitude"]
            .isna()
            .sum()
        )
    },
    {
        "Check": "Coordinate pair mismatch",
        "Expected": 0,
        "Actual": coordinate_pair_mismatch_disk
    },
    {
        "Check": "Missing speed_limit",
        "Expected": 3,
        "Actual": (
            collisions_disk["speed_limit"]
            .isna()
            .sum()
        )
    },
    {
        "Check": "Missing local authority name",
        "Expected": 0,
        "Actual": (
            collisions_disk[
                "local_authority_ons_district_name"
            ]
            .isna()
            .sum()
        )
    }
])

content_qa["Passed"] = (
    content_qa["Expected"] ==
    content_qa["Actual"]
)

display(content_qa)

print(
    "Các speed limit đọc lại từ CSV:",
    sorted(actual_speed_limits_disk)
)

if actual_speed_limits_disk != {
    20, 30, 40, 50, 60, 70
}:
    raise ValueError(
        "speed_limit sau khi đọc lại không hợp lệ."
    )

if not content_qa["Passed"].all():
    raise ValueError(
        "Nội dung Collisions Clean không đạt QA."
    )

print("Nội dung Collisions Clean hợp lệ")

,Check,Expected,Actual,Passed
0,Invalid ISO date,0,0,True
1,Invalid HH:MM time,0,0,True
2,Years outside 2021–2025,0,0,True
3,Missing coordinate pairs,53,53,True
4,Coordinate pair mismatch,0,0,True
5,Missing speed_limit,3,3,True
6,Missing local authority name,0,0,True


Các speed limit đọc lại từ CSV: [20, 30, 40, 50, 60, 70]
Nội dung Collisions Clean hợp lệ


In [120]:
# Cell 32 — Tổng kết ba file Clean
final_clean_summary = pd.DataFrame([
    {
        "Table": "Collisions",
        "Rows": len(collisions_disk),
        "Columns": len(collisions_disk.columns),
        "Natural_Key": "collision_index",
        "Output_File": COLLISIONS_CLEAN_PATH.name,
        "Status": "PASS"
    },
    {
        "Table": "Vehicles",
        "Rows": len(vehicles_disk),
        "Columns": len(vehicles_disk.columns),
        "Natural_Key": (
            "collision_index + vehicle_reference"
        ),
        "Output_File": VEHICLES_CLEAN_PATH.name,
        "Status": "PASS"
    },
    {
        "Table": "Casualties",
        "Rows": len(casualties_disk),
        "Columns": len(casualties_disk.columns),
        "Natural_Key": (
            "collision_index + casualty_reference"
        ),
        "Output_File": CASUALTIES_CLEAN_PATH.name,
        "Status": "PASS"
    }
])

display(final_clean_summary)

print("RAW → CLEAN HOÀN TẤT")
print("Ba file đã sẵn sàng để thiết kế dimensional model")

,Table,Rows,Columns,Natural_Key,Output_File,Status
0,Collisions,513801,18,collision_index,collisions_clean.csv,PASS
1,Vehicles,937265,8,collision_index + vehicle_reference,vehicles_clean.csv,PASS
2,Casualties,652821,8,collision_index + casualty_reference,casualties_clean.csv,PASS


RAW → CLEAN HOÀN TẤT
Ba file đã sẵn sàng để thiết kế dimensional model
